In [1]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    build_feature_combos,
    compute_batch_size,
    detect_device,
    load_split_bundle,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Mounted at /content/drive


## Load data

In [2]:
DATA_SET = 'rand_D'

df_train, df_val, df_test = load_split_bundle(CLEAN_DATA, DATA_SET)

OUTPUT_DIR = make_run_dir()

## GPU auto-detect

In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

A100-80GB  |  VRAM: 85 GB  |  MAX_BATCH=65,536  |  dtype=torch.bfloat16


## Hyperparameters

In [4]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BATCH_SIZE = compute_batch_size(len(df_train), CFG['MAX_BATCH'])
BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

steps_per_epoch = len(df_train) // BATCH_SIZE
print(f'MAX_BATCH={CFG["MAX_BATCH"]:,}  adaptive BATCH_SIZE={BATCH_SIZE:,}  '
      f'INIT_LR={INIT_LR:.6f}  n_train={len(df_train):,}  '
      f'steps/epoch~{steps_per_epoch}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'WARMUP={WARMUP_EPOCHS} epochs')

MAX_BATCH=65,536  adaptive BATCH_SIZE=16,384  INIT_LR=0.002000  n_train=843,153  steps/epoch~51
MAX_EPOCHS=100  PATIENCE=30  WARMUP=5 epochs


## Feature definitions

In [5]:
FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'iv_lag', 'd_iv_lag', 'vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

feature_combos = build_feature_combos(FEATURES_3, EXTRA_FEATURES, max_extra=3)

n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')

Total feature combinations: 130
  Base (3F):  1
  3F + 1:     9
  3F + 2:     36
  3F + 3:     84


## Pre-allocate on GPU

In [6]:
gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Data on GPU  |  VRAM used: 0.53 GB / 85 GB  |  Free: 84.6 GB
Train: 843,153  Val: 240,901  Test: 120,451  Features: 12


## Analytic benchmark

In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 8.3745  RMSE = 0.008338
Coefficients: a = -0.165665, b = -0.000516, c = 0.032673


## Train all feature combinations

In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    nan_mask_tr=gpu_data['nan_mask_tr'],
    nan_mask_va=gpu_data['nan_mask_va'],
    nan_mask_ytr=gpu_data['nan_mask_ytr'],
    nan_mask_yva=gpu_data['nan_mask_yva'],
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

sep = "=" * 70
print('\n' + sep)
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(sep + '\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 130 MODELS
  GPU: A100-80GB  |  Batch: 16,384  |  AMP: True  |  Max epochs: 100

  [  1/130] 3F                             SSE=10.3267  Gain=-23.3%  ep=100  18.9s  elapsed=0.3min
  [ 25/130] 3F+iv_lag+rho                  SSE=10.5686  Gain=-26.2%  ep=100  12.2s  elapsed=5.2min
  [ 50/130] 3F+vix_lag+iv_lag+gamma        SSE=7.8173  Gain=+6.7%  ep=100  12.1s  elapsed=10.4min
  [ 75/130] 3F+iv_lag+d_iv_lag+vix_mom_lag SSE=11.5661  Gain=-38.1%  ep=100  12.0s  elapsed=15.5min
  [100/130] 3F+d_iv_lag+vix_mom_lag+rho    SSE=12.4134  Gain=-48.2%  ep=100  11.9s  elapsed=20.6min
  [125/130] 3F+vix_mom+theta+rho           SSE=11.4450  Gain=-36.7%  ep=100  12.5s  elapsed=25.7min
  [130/130] 3F+theta+vega+rho              SSE=12.2002  Gain=-45.7%  ep=100  12.5s  elapsed=26.7min

Done: 26.7 min for 130 models (avg 12.3s/model)


## Results summary

In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,8.374537,0.000070,0.008338,0.003523,0.000748,0.001885,0.125011,None,None,None
1,3F,10.326705,0.000086,0.009259,0.005202,-0.000461,0.003535,-0.078955,13.1s,-23.31%,None
2,3F+vix_lag,22.640827,0.000188,0.013710,0.009333,0.001672,0.006792,-1.365560,12.3s,-170.35%,-119.25%
3,3F+iv_lag,16.829990,0.000140,0.011821,0.008364,-0.001261,0.006219,-0.758432,12.3s,-100.97%,25.67%
4,3F+d_iv_lag,19.636833,0.000163,0.012768,0.008646,0.002018,0.006167,-1.051696,12.0s,-134.48%,-16.68%
...,...,...,...,...,...,...,...,...,...,...,...
127,3F+gamma+theta+vega,12.554137,0.000104,0.010209,0.006640,0.001412,0.004871,-0.311682,12.1s,-49.91%,-1.99%
128,3F+gamma+theta+rho,12.304901,0.000102,0.010107,0.006671,-0.000069,0.004945,-0.285641,12.3s,-46.93%,1.99%
129,3F+gamma+vega+rho,14.577847,0.000121,0.011001,0.007297,0.000308,0.005520,-0.523123,12.5s,-74.07%,-18.47%
130,3F+theta+vega+rho,12.200168,0.000101,0.010064,0.006471,-0.000192,0.004782,-0.274698,12.5s,-45.68%,16.31%


## Top 10 by Gain vs Hull-White

In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,3F+vix_lag+iv_lag+gamma,6,7.8173,0.008056,6.65,12.1
1,3F+iv_lag+gamma+vega,6,8.1275,0.008214,2.95,12.2
2,3F+vix_lag+iv_lag+vega,6,8.2495,0.008276,1.49,12.7
3,3F+iv_lag+gamma+rho,6,8.4325,0.008367,-0.69,12.2
4,3F+iv_lag+gamma+theta,6,8.5272,0.008414,-1.82,12.2
5,3F+vix_lag+iv_lag+d_iv_lag,6,9.1192,0.008701,-8.89,11.9
6,3F+vix_lag+iv_lag+vix_mom,6,9.1506,0.008716,-9.27,12.3
7,3F+vix_lag+iv_lag+theta,6,9.2746,0.008775,-10.75,12.1
8,3F+iv_lag+vix_mom_lag+theta,6,9.3054,0.008789,-11.11,12.5
9,3F+iv_lag+theta,5,9.3156,0.008794,-11.24,12.1


## Best per group (3F, +1, +2, +3)

In [11]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f"{nf_label}: {best['combo_name']}")
    print(f"    SSE={best['SSE']:.4f}  RMSE={best['RMSE']:.6f}  Gain={best['Gain_vs_HW_%']:.2f}%\n")

3F (base): 3F
    SSE=10.3267  RMSE=0.009259  Gain=-23.31%

+1 (4F): 3F+gamma
    SSE=15.9891  RMSE=0.011521  Gain=-90.92%

+2 (5F): 3F+iv_lag+theta
    SSE=9.3156  RMSE=0.008794  Gain=-11.24%

+3 (6F): 3F+vix_lag+iv_lag+gamma
    SSE=7.8173  RMSE=0.008056  Gain=6.65%



## Summary statistics

In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 26.6 min (0.44 hr)
Models trained: 130
Best overall: 3F+vix_lag+iv_lag+gamma (Gain=6.65%)


## Cleanup

In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.69 GB / 85 GB
Total training time: 26.6 min for 130 models
